# Exp 008 Long Context Native SED

## Idea

Move the native branch from independent `5s` clip classification to a long-context soundscape model:

- input: `20s` waveform context
- output: `4` aligned `5s` prediction windows
- validation: overlap-aware aggregation back to row-level `row_id` predictions

## Why this matters

Our current native branch improved steadily from `0.647 -> 0.737 -> 0.758` on Kaggle, but those gains came mostly from stronger postprocessing and fold ensembling. The next likely jump should come from modeling more soundscape context directly.

## Hypothesis

A `20s -> 4 x 5s` native SED branch will transfer better to hidden soundscapes than the current short-context native model, even before pseudo-labeling.


## Notebook Notes

This first `exp_008` version intentionally keeps the recipe simple:

- supervised training on labeled soundscapes only
- optional soundscape background mixing
- partial initialization from `exp_002`
- no pseudo-labeling yet
- no external Perch features

If this branch looks promising, the next step will be to reuse the `exp_007` inference layer on top of its validation exports and later on Kaggle.


In [8]:
from __future__ import annotations

import json
import random
import sys
from collections import OrderedDict, defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / 'data' / 'birdclef-2026').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not resolve PROJECT_ROOT with data/birdclef-2026')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from birdclef2026.reference_eval import (  # noqa: E402
    ReferenceModelConfig,
    extract_state_dict,
    safe_load_checkpoint,
    score_macro_auc,
)


@dataclass
class LongContextConfig:
    seed: int = 42
    sample_rate: int = 32_000
    context_duration: float = 20.0
    target_duration: float = 5.0
    context_hop: float = 5.0
    batch_size: int = 6
    num_workers: int = 0
    epochs: int = 6
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    use_amp: bool = True
    fold: int = 2
    n_folds: int = 5
    use_background_mixing: bool = True
    bg_mix_prob: float = 0.50
    bg_snr_db_min: float = 10.0
    bg_snr_db_max: float = 24.0
    bg_target_weight: float = 0.25
    soundscape_cache_size: int = 24
    max_train_contexts: int | None = None
    max_valid_contexts: int | None = None
    export_validation_outputs: bool = True
    init_checkpoint_name: str = 'best_model.pt'

    @property
    def context_samples(self) -> int:
        return int(self.sample_rate * self.context_duration)

    @property
    def target_samples(self) -> int:
        return int(self.sample_rate * self.target_duration)

    @property
    def num_output_steps(self) -> int:
        return int(round(self.context_duration / self.target_duration))


CFG = LongContextConfig()
DATA_DIR = PROJECT_ROOT / 'data' / 'birdclef-2026'
TRAIN_SOUNDSCAPE_DIR = DATA_DIR / 'train_soundscapes'
EXP002_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_002_train_audio_reproduction'
INIT_CHECKPOINT_PATH = EXP002_DIR / CFG.init_checkpoint_name
BASE_OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_008_long_context_native_sed'
OUTPUT_DIR = BASE_OUTPUT_DIR / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
amp_enabled = CFG.use_amp and device.type == 'cuda'


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

{
    'project_root': str(PROJECT_ROOT),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'init_checkpoint_path': str(INIT_CHECKPOINT_PATH),
    'output_dir': str(OUTPUT_DIR),
    **asdict(CFG),
}


{'project_root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026',
 'device': 'mps',
 'amp_enabled': False,
 'init_checkpoint_path': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_002_train_audio_reproduction/best_model.pt',
 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_008_long_context_native_sed/fold_02',
 'seed': 42,
 'sample_rate': 32000,
 'context_duration': 20.0,
 'target_duration': 5.0,
 'context_hop': 5.0,
 'batch_size': 6,
 'num_workers': 0,
 'epochs': 6,
 'learning_rate': 0.0002,
 'weight_decay': 0.0001,
 'use_amp': True,
 'fold': 2,
 'n_folds': 5,
 'use_background_mixing': True,
 'bg_mix_prob': 0.5,
 'bg_snr_db_min': 10.0,
 'bg_snr_db_max': 24.0,
 'bg_target_weight': 0.25,
 'soundscape_cache_size': 24,
 'max_train_contexts': None,
 'max_valid_contexts': None,
 'export_validation_outputs': True,
 'init_checkpoint_name': 'best_model.pt'}

In [9]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
species = sample_sub.columns[1:].tolist()
label_to_idx = {label: idx for idx, label in enumerate(species)}


def parse_soundscape_labels(value) -> list[str]:
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str) -> dict:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def encode_multi_hot(labels: list[str]) -> np.ndarray:
    target = np.zeros(len(species), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


def build_context_frame(segment_df: pd.DataFrame, cfg: LongContextConfig) -> pd.DataFrame:
    rows = []
    hop_sec = int(cfg.context_hop)
    target_sec = int(cfg.target_duration)
    context_sec = int(cfg.context_duration)
    for filename, file_df in segment_df.groupby('filename', sort=False):
        ordered = file_df.sort_values('end_sec').reset_index(drop=True)
        if ordered.empty:
            continue
        file_duration = int(ordered['end_sec'].max())
        max_start = file_duration - context_sec
        if max_start < 0:
            continue
        end_lookup = {int(row.end_sec): row for row in ordered.itertuples(index=False)}
        for start_sec in range(0, max_start + 1, hop_sec):
            window_end_secs = [start_sec + target_sec * (idx + 1) for idx in range(cfg.num_output_steps)]
            if any(end_sec not in end_lookup for end_sec in window_end_secs):
                continue
            window_rows = [end_lookup[end_sec] for end_sec in window_end_secs]
            rows.append({
                'filename': filename,
                'file_path': Path(window_rows[0].file_path),
                'context_start_sec': int(start_sec),
                'context_end_sec': int(start_sec + context_sec),
                'site': window_rows[0].site,
                'hour_utc': int(window_rows[0].hour_utc),
                'window_end_secs': window_end_secs,
                'window_row_ids': [row.row_id for row in window_rows],
                'window_label_lists': [list(row.label_list) for row in window_rows],
            })
    return pd.DataFrame(rows)


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)
sc_clean['file_path'] = sc_clean['filename'].apply(lambda name: TRAIN_SOUNDSCAPE_DIR / name)
sc_clean = sc_clean[sc_clean['file_path'].map(Path.exists)].reset_index(drop=True)
sc_clean = sc_clean.sort_values(['filename', 'end_sec']).reset_index(drop=True)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
full_df = sc_clean[sc_clean['filename'].isin(full_files)].copy().reset_index(drop=True)

n_group_splits = min(CFG.n_folds, max(2, len(full_files)))
gkf = GroupKFold(n_splits=n_group_splits)
full_splits = list(gkf.split(full_df, groups=full_df['filename']))
train_full_idx, valid_full_idx = full_splits[CFG.fold % len(full_splits)]
valid_files = set(full_df.iloc[valid_full_idx]['filename'].tolist())

train_segment_df = (
    full_df[~full_df['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)
valid_segment_df = (
    full_df[full_df['filename'].isin(valid_files)]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=True)
)

train_context_df = build_context_frame(train_segment_df, CFG)
valid_context_df = build_context_frame(valid_segment_df, CFG)

if CFG.max_train_contexts is not None:
    train_context_df = train_context_df.head(CFG.max_train_contexts).reset_index(drop=True)
if CFG.max_valid_contexts is not None:
    valid_context_df = valid_context_df.head(CFG.max_valid_contexts).reset_index(drop=True)

valid_target_df = pd.DataFrame(
    np.stack([encode_multi_hot(labels) for labels in valid_segment_df['label_list']], axis=0),
    index=valid_segment_df['row_id'],
    columns=species,
)
valid_target_df.index.name = 'row_id'

{
    'fold': int(CFG.fold),
    'n_group_splits': int(len(full_splits)),
    'fully_labeled_files': int(len(full_files)),
    'train_soundscape_files': int(train_segment_df['filename'].nunique()),
    'valid_soundscape_files': int(valid_segment_df['filename'].nunique()),
    'train_contexts': int(len(train_context_df)),
    'valid_contexts': int(len(valid_context_df)),
    'target_rows_in_valid': int(len(valid_target_df)),
    'context_duration_sec': int(CFG.context_duration),
    'target_duration_sec': int(CFG.target_duration),
    'outputs_per_context': int(CFG.num_output_steps),
}


{'fold': 2,
 'n_group_splits': 5,
 'fully_labeled_files': 59,
 'train_soundscape_files': 47,
 'valid_soundscape_files': 12,
 'train_contexts': 423,
 'valid_contexts': 108,
 'target_rows_in_valid': 144,
 'context_duration_sec': 20,
 'target_duration_sec': 5,
 'outputs_per_context': 4}

In [10]:
class LongContextSoundscapeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, bg_frame: pd.DataFrame, cfg: LongContextConfig, label_to_idx: dict[str, int], is_train: bool):
        self.frame = frame.reset_index(drop=True)
        self.bg_frame = bg_frame.reset_index(drop=True) if len(bg_frame) else frame.reset_index(drop=True)
        self.cfg = cfg
        self.label_to_idx = label_to_idx
        self.is_train = is_train
        self.cache: OrderedDict[str, np.ndarray] = OrderedDict()

    def __len__(self) -> int:
        return len(self.frame)

    def _load_soundscape(self, file_path: Path) -> np.ndarray:
        key = str(file_path)
        if key in self.cache:
            audio = self.cache.pop(key)
            self.cache[key] = audio
            return audio

        audio, sr = sf.read(file_path, dtype='float32', always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != self.cfg.sample_rate:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=self.cfg.sample_rate)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        self.cache[key] = audio
        while len(self.cache) > self.cfg.soundscape_cache_size:
            self.cache.popitem(last=False)
        return audio

    def _slice_context(self, audio: np.ndarray, start_sec: int) -> np.ndarray:
        start = int(start_sec * self.cfg.sample_rate)
        end = start + self.cfg.context_samples
        clip = audio[start:end]
        if len(clip) < self.cfg.context_samples:
            clip = np.pad(clip, (0, self.cfg.context_samples - len(clip)))
        return clip.astype(np.float32)

    def _encode_matrix(self, label_lists: list[list[str]]) -> np.ndarray:
        target = np.zeros((self.cfg.num_output_steps, len(self.label_to_idx)), dtype=np.float32)
        for step_idx, labels in enumerate(label_lists):
            for label in labels:
                class_idx = self.label_to_idx.get(label)
                if class_idx is not None:
                    target[step_idx, class_idx] = 1.0
        return target

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        audio = self._slice_context(self._load_soundscape(Path(row['file_path'])), int(row['context_start_sec']))
        target = self._encode_matrix(row['window_label_lists'])

        bg_audio = np.zeros_like(audio)
        bg_target = np.zeros_like(target)
        if self.is_train and self.cfg.use_background_mixing and len(self.bg_frame) > 1:
            bg_row = self.bg_frame.iloc[np.random.randint(0, len(self.bg_frame))]
            bg_audio = self._slice_context(self._load_soundscape(Path(bg_row['file_path'])), int(bg_row['context_start_sec']))
            bg_target = self._encode_matrix(bg_row['window_label_lists'])

        site = '' if pd.isna(row['site']) else str(row['site'])
        return {
            'audio': torch.from_numpy(audio),
            'target': torch.from_numpy(target),
            'bg_audio': torch.from_numpy(bg_audio),
            'bg_target': torch.from_numpy(bg_target),
            'row_ids': list(row['window_row_ids']),
            'filename': row['filename'],
            'window_end_secs': list(row['window_end_secs']),
            'site': site,
            'hour_utc': int(row['hour_utc']),
        }


def context_collate(batch: list[dict]) -> dict:
    return {
        'audio': torch.stack([item['audio'] for item in batch], dim=0),
        'target': torch.stack([item['target'] for item in batch], dim=0),
        'bg_audio': torch.stack([item['bg_audio'] for item in batch], dim=0),
        'bg_target': torch.stack([item['bg_target'] for item in batch], dim=0),
        'row_ids': [item['row_ids'] for item in batch],
        'filename': [item['filename'] for item in batch],
        'window_end_secs': [item['window_end_secs'] for item in batch],
        'site': [item['site'] for item in batch],
        'hour_utc': torch.tensor([item['hour_utc'] for item in batch], dtype=torch.int64),
    }


def apply_background_mix(audio: torch.Tensor, bg_audio: torch.Tensor, bg_target: torch.Tensor, cfg: LongContextConfig) -> tuple[torch.Tensor, torch.Tensor]:
    if not cfg.use_background_mixing:
        return audio, torch.zeros_like(bg_target)

    device = audio.device
    batch_size = audio.size(0)
    has_bg = (bg_target.sum(dim=(1, 2), keepdim=True) > 0).float()
    mix_mask = (torch.rand(batch_size, 1, 1, device=device) < cfg.bg_mix_prob).float() * has_bg

    rms_audio = torch.sqrt(torch.mean(audio ** 2, dim=1, keepdim=True) + 1e-9)
    rms_bg = torch.sqrt(torch.mean(bg_audio ** 2, dim=1, keepdim=True) + 1e-9)
    snr_db = torch.empty(batch_size, 1, device=device).uniform_(cfg.bg_snr_db_min, cfg.bg_snr_db_max)
    scale = rms_audio / (rms_bg * (10.0 ** (snr_db / 20.0)) + 1e-9)

    mixed_audio = torch.clamp(audio + bg_audio * scale, -1.0, 1.0)
    mixed_audio = mix_mask.squeeze(-1) * mixed_audio + (1.0 - mix_mask.squeeze(-1)) * audio
    soft_targets = bg_target * mix_mask * cfg.bg_target_weight
    return mixed_audio, soft_targets


train_dataset = LongContextSoundscapeDataset(train_context_df, train_context_df, CFG, label_to_idx, is_train=True)
valid_dataset = LongContextSoundscapeDataset(valid_context_df, valid_context_df, CFG, label_to_idx, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=context_collate,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=context_collate,
)

{
    'train_batches': int(len(train_loader)),
    'valid_batches': int(len(valid_loader)),
    'train_dataset_size': int(len(train_dataset)),
    'valid_dataset_size': int(len(valid_dataset)),
}


{'train_batches': 71,
 'valid_batches': 18,
 'train_dataset_size': 423,
 'valid_dataset_size': 108}

In [11]:
class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class TemporalWindowHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, num_steps: int, dropout: float = 0.1):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Conv1d(feat_dim, feat_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(num_steps)
        self.classifier = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x = self.pre(x)
        x = self.pool(x)
        windowwise_logit = self.classifier(x).transpose(1, 2)
        windowwise_prob = torch.sigmoid(windowwise_logit)
        clipwise_prob = windowwise_prob.max(dim=1).values
        return {
            'windowwise_logit': windowwise_logit,
            'windowwise_prob': windowwise_prob,
            'clipwise_prob': clipwise_prob,
        }


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: ReferenceModelConfig):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)

        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class WaveformLongContextSED(nn.Module):
    def __init__(self, cfg_native: LongContextConfig, num_classes: int):
        super().__init__()
        self.ref_cfg = ReferenceModelConfig(num_classes=num_classes, chunk_duration=cfg_native.context_duration)
        self.mel = MelSpectrogramTransform(self.ref_cfg)
        self.backbone = timm.create_model(
            self.ref_cfg.backbone,
            pretrained=False,
            in_chans=self.ref_cfg.in_channels,
            features_only=False,
            global_pool='',
            num_classes=0,
            drop_path_rate=self.ref_cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.freq_pool = GEMFreqPool(p_init=self.ref_cfg.gem_p_init)
        self.head = TemporalWindowHead(feat_dim, num_classes, cfg_native.num_output_steps, self.ref_cfg.dropout)

    def forward(self, waveforms: torch.Tensor) -> dict[str, torch.Tensor]:
        mel = self.mel(waveforms)
        features = self.backbone(mel)
        temporal = self.freq_pool(features)
        return self.head(temporal)


class WindowBCE(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor, soft_targets: torch.Tensor | None = None) -> torch.Tensor:
        combined_targets = targets if soft_targets is None else torch.maximum(targets, soft_targets)
        return self.criterion(logits, combined_targets)


def initialize_from_clip_checkpoint(model: WaveformLongContextSED, checkpoint_path: Path) -> dict:
    if not checkpoint_path.exists():
        return {'loaded': False, 'reason': 'checkpoint_missing'}

    ckpt = safe_load_checkpoint(checkpoint_path)
    state_dict = extract_state_dict(ckpt)
    remapped = {}
    for key, value in state_dict.items():
        if key.startswith('mel.'):
            remapped[key] = value
        elif key.startswith('model.backbone.'):
            remapped['backbone.' + key[len('model.backbone.'):]] = value
        elif key.startswith('model.gem_pool.'):
            remapped['freq_pool.' + key[len('model.gem_pool.'):]] = value

    load_result = model.load_state_dict(remapped, strict=False)
    return {
        'loaded': True,
        'source': str(checkpoint_path),
        'missing_keys': len(load_result.missing_keys),
        'unexpected_keys': len(load_result.unexpected_keys),
    }


model = WaveformLongContextSED(CFG, len(species)).to(device)
init_summary = initialize_from_clip_checkpoint(model, INIT_CHECKPOINT_PATH)
criterion = WindowBCE()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))

init_summary


{'loaded': True,
 'source': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_002_train_audio_reproduction/best_model.pt',
 'missing_keys': 4,
 'unexpected_keys': 0}

In [12]:
def train_one_epoch(loader: DataLoader, model: nn.Module, criterion: WindowBCE, optimizer, cfg: LongContextConfig) -> float:
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in tqdm(loader, desc='train', leave=False):
        audio = batch['audio'].to(device)
        targets = batch['target'].to(device)
        bg_audio = batch['bg_audio'].to(device)
        bg_target = batch['bg_target'].to(device)

        mixed_audio, soft_targets = apply_background_mix(audio, bg_audio, bg_target, cfg)

        optimizer.zero_grad(set_to_none=True)
        with (torch.amp.autocast(device_type='cuda', enabled=amp_enabled) if amp_enabled else nullcontext()):
            output = model(mixed_audio)
            loss = criterion(output['windowwise_logit'], targets, soft_targets)
        loss.backward()
        optimizer.step()

        batch_size = audio.size(0)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_examples += batch_size

    return total_loss / max(1, total_examples)


@torch.no_grad()
def validate_one_epoch(loader: DataLoader, model: nn.Module, criterion: WindowBCE, valid_targets: pd.DataFrame) -> tuple[float, dict, pd.DataFrame]:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    row_logits: dict[str, list[np.ndarray]] = defaultdict(list)
    row_meta: dict[str, dict] = {}

    for batch in tqdm(loader, desc='valid', leave=False):
        audio = batch['audio'].to(device)
        targets = batch['target'].to(device)
        output = model(audio)
        loss = criterion(output['windowwise_logit'], targets)

        logits = output['windowwise_logit'].detach().cpu().numpy().astype(np.float32)
        batch_size = audio.size(0)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_examples += batch_size

        for sample_idx, row_ids in enumerate(batch['row_ids']):
            for step_idx, row_id in enumerate(row_ids):
                row_logits[row_id].append(logits[sample_idx, step_idx])
                row_meta[row_id] = {
                    'row_id': row_id,
                    'filename': batch['filename'][sample_idx],
                    'end_sec': int(batch['window_end_secs'][sample_idx][step_idx]),
                    'site': batch['site'][sample_idx],
                    'hour_utc': int(batch['hour_utc'][sample_idx].item()),
                }

    ordered_row_ids = valid_targets.index.tolist()
    y_true = valid_targets.loc[ordered_row_ids].values.astype(np.float32)
    y_pred = np.zeros_like(y_true, dtype=np.float32)
    counts = []
    meta_rows = []
    for row_idx, row_id in enumerate(ordered_row_ids):
        values = row_logits.get(row_id, [])
        counts.append(len(values))
        if values:
            mean_logit = np.mean(np.stack(values, axis=0), axis=0)
            y_pred[row_idx] = 1.0 / (1.0 + np.exp(-mean_logit))
        meta = row_meta.get(row_id, {'row_id': row_id, 'filename': None, 'end_sec': None, 'site': None, 'hour_utc': -1})
        meta['coverage_count'] = len(values)
        meta_rows.append(meta)

    metrics = score_macro_auc(y_true, y_pred, species)
    metrics['coverage_mean'] = float(np.mean(counts)) if counts else 0.0
    metrics['coverage_min'] = int(np.min(counts)) if counts else 0
    metrics['coverage_max'] = int(np.max(counts)) if counts else 0

    export_df = pd.DataFrame(meta_rows)
    for idx, label in enumerate(species):
        export_df[f'true__{label}'] = y_true[:, idx].astype(np.int8)
        export_df[f'pred__{label}'] = y_pred[:, idx].astype(np.float32)

    return total_loss / max(1, total_examples), metrics, export_df


def save_checkpoint(path: Path, epoch: int, model: nn.Module, optimizer, scheduler, history: list[dict], best_metric: float) -> None:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_metric': best_metric,
        'stage': 'exp_008_long_context_native_sed',
    }, path)


In [13]:
RUN_TRAINING = True

history = []
best_metric = float('-inf')
best_epoch = None
best_valid_df = None

if RUN_TRAINING:
    for epoch in range(1, CFG.epochs + 1):
        train_loss = train_one_epoch(train_loader, model, criterion, optimizer, CFG)
        valid_loss, valid_metrics, valid_export_df = validate_one_epoch(valid_loader, model, criterion, valid_target_df)
        scheduler.step()

        row = {
            'epoch': epoch,
            'train_loss': float(train_loss),
            'macro_auc': float(valid_metrics['macro_auc']),
            'scored_classes': int(valid_metrics['scored_classes']),
            'skipped_no_positive': int(valid_metrics['skipped_no_positive']),
            'skipped_no_negative': int(valid_metrics['skipped_no_negative']),
            'coverage_mean': float(valid_metrics['coverage_mean']),
            'coverage_min': int(valid_metrics['coverage_min']),
            'coverage_max': int(valid_metrics['coverage_max']),
            'valid_loss': float(valid_loss),
            'learning_rate': float(optimizer.param_groups[0]['lr']),
        }
        history.append(row)
        print(row)

        save_checkpoint(OUTPUT_DIR / 'last_model.pt', epoch, model, optimizer, scheduler, history, best_metric)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)

        if row['macro_auc'] > best_metric:
            best_metric = row['macro_auc']
            best_epoch = epoch
            best_valid_df = valid_export_df.copy()
            save_checkpoint(OUTPUT_DIR / 'best_model.pt', epoch, model, optimizer, scheduler, history, best_metric)
            if CFG.export_validation_outputs:
                valid_export_df.to_csv(OUTPUT_DIR / 'best_valid_predictions.csv', index=False)
                meta_cols = ['row_id', 'filename', 'end_sec', 'site', 'hour_utc', 'coverage_count']
                valid_export_df[meta_cols].to_csv(OUTPUT_DIR / 'best_valid_meta.csv', index=False)
                true_cols = [f'true__{label}' for label in species]
                pred_cols = [f'pred__{label}' for label in species]
                np.savez_compressed(
                    OUTPUT_DIR / 'best_valid_outputs.npz',
                    y_true=valid_export_df[true_cols].values.astype(np.float32),
                    y_pred=valid_export_df[pred_cols].values.astype(np.float32),
                )

    snapshot = {
        'experiment_id': 'exp_008',
        'experiment_name': 'long_context_native_sed',
        'fold': int(CFG.fold),
        'best_epoch': int(best_epoch) if best_epoch is not None else None,
        'best_macro_auc': float(best_metric) if best_epoch is not None else None,
        'output_dir': str(OUTPUT_DIR),
        'context_duration': float(CFG.context_duration),
        'target_duration': float(CFG.target_duration),
        'num_output_steps': int(CFG.num_output_steps),
        'train_contexts': int(len(train_context_df)),
        'valid_contexts': int(len(valid_context_df)),
    }
    (OUTPUT_DIR / 'result_snapshot.json').write_text(json.dumps(snapshot, indent=2))
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))
else:
    print('RUN_TRAINING is False. Review config, then switch it to True for the first fold run.')
    print({
        'output_dir': str(OUTPUT_DIR),
        'train_contexts': int(len(train_context_df)),
        'valid_contexts': int(len(valid_context_df)),
        'init_checkpoint_exists': INIT_CHECKPOINT_PATH.exists(),
        'device': str(device),
    })


train:   0%|          | 0/71 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [6, 1251, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [3, 1251, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero el

valid:   0%|          | 0/18 [00:00<?, ?it/s]

/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_16336/2065872706.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'pred__{label}'] = y_pred[:, idx].astype(np.float32)
/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_16336/2065872706.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'true__{label}'] = y_true[:, idx].astype(np.int8)


{'epoch': 1, 'train_loss': 0.11930209813071481, 'macro_auc': 0.7694400463587634, 'scored_classes': 35, 'skipped_no_positive': 199, 'skipped_no_negative': 0, 'coverage_mean': 3.0, 'coverage_min': 1, 'coverage_max': 4, 'valid_loss': 0.05090202587760157, 'learning_rate': 0.00018660254037844388}


...

train:   0%|          | 0/71 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [6, 1251, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [3, 1251, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero el

valid:   0%|          | 0/18 [00:00<?, ?it/s]

/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_16336/2065872706.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'pred__{label}'] = y_pred[:, idx].astype(np.float32)
/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_16336/2065872706.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  export_df[f'true__{label}'] = y_true[:, idx].astype(np.int8)


{'epoch': 6, 'train_loss': 0.036179032784404486, 'macro_auc': 0.7797011798763266, 'scored_classes': 35, 'skipped_no_positive': 199, 'skipped_no_negative': 0, 'coverage_mean': 3.0, 'coverage_min': 1, 'coverage_max': 4, 'valid_loss': 0.04428406794452005, 'learning_rate': 0.0}
Snapshot:
{
  "experiment_id": "exp_008",
  "experiment_name": "long_context_native_sed",
  "fold": 2,
  "best_epoch": 5,
  "best_macro_auc": 0.8094790078756339,
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_008_long_context_native_sed/fold_02",
  "context_duration": 20.0,
  "target_duration": 5.0,
  "num_output_steps": 4,
  "train_contexts": 423,
  "valid_contexts": 108
}


## Results And Notes

After a real run, use this section to record:

- best validation macro ROC-AUC
- scored classes
- row-level coverage statistics from overlap aggregation
- whether the longer context helps enough to justify an `exp_008` Kaggle submission

A good first smoke test is:

```python
CFG.max_train_contexts = 32
CFG.max_valid_contexts = 16
CFG.epochs = 1
RUN_TRAINING = True
```


In [14]:
if (OUTPUT_DIR / 'result_snapshot.json').exists():
    snapshot = json.loads((OUTPUT_DIR / 'result_snapshot.json').read_text())
    print('Snapshot:')
    print(json.dumps(snapshot, indent=2))

    history_path = OUTPUT_DIR / 'history.csv'
    if history_path.exists():
        display(pd.read_csv(history_path))
else:
    print('No completed run found yet for this fold.')


Snapshot:
{
  "experiment_id": "exp_008",
  "experiment_name": "long_context_native_sed",
  "fold": 2,
  "best_epoch": 5,
  "best_macro_auc": 0.8094790078756339,
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_008_long_context_native_sed/fold_02",
  "context_duration": 20.0,
  "target_duration": 5.0,
  "num_output_steps": 4,
  "train_contexts": 423,
  "valid_contexts": 108
}


,epoch,train_loss,macro_auc,scored_classes,skipped_no_positive,skipped_no_negative,coverage_mean,coverage_min,coverage_max,valid_loss,learning_rate
0,1,0.119302,0.769440,35,199,0,3.0,1,4,0.050902,0.000187
1,2,0.048271,0.791088,35,199,0,3.0,1,4,0.045590,0.000150
2,3,0.042746,0.796148,35,199,0,3.0,1,4,0.045901,0.000100
3,4,0.040694,0.786542,35,199,0,3.0,1,4,0.044103,0.000050
4,5,0.038396,0.809479,35,199,0,3.0,1,4,0.042924,0.000013
5,6,0.036179,0.779701,35,199,0,3.0,1,4,0.044284,0.000000


## Next Steps

If `exp_008` shows a real local gain, the next sequence should be:

1. apply the existing `exp_007` priors and texture-aware smoothing on top of `exp_008` validation exports
2. build a dedicated Kaggle submission notebook for the strongest `exp_008` variant
3. if long context is promising but still insufficient, continue to `exp_009` noisy-student pseudo-labeling
